# Implement Quantum Teleportation algorithm in Python
**Computer Laboratory II (417526) – Quantum AI – Practical 4**

Install once (Linux terminal):
```
pip install qiskit qiskit-aer matplotlib pylatexenc
```
Runs on the local Aer simulator – no IBM account or API token needed (the lab manual's `execute` / `IBMQ` / `qiskit.aqua` / `qiskit.ignis` code was removed from current Qiskit, so this notebook uses the current API).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

import qiskit, qiskit_aer
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

sim = AerSimulator()
print("qiskit", qiskit.__version__, "| qiskit-aer", qiskit_aer.__version__)

def run(qc, shots=1024, backend=sim, **kw):
    # Transpile for the backend, run, and return the counts dictionary.
    t = transpile(qc, backend, **kw)
    return backend.run(t, shots=shots).result().get_counts()

## Practical 4 – Quantum Teleportation
**Idea:** Alice teleports the state of qubit q0 to Bob's qubit q2 using an entangled pair (q1, q2) and two classical bits.

1. Create a Bell pair between q1 and q2 (H on q1, CNOT q1→q2)
2. Alice entangles q0 with q1 (CNOT q0→q1, H on q0) and measures q0, q1
3. Bob applies **X** (if q1 measured 1) and **Z** (if q0 measured 1) to q2: CX(q1→q2), CZ(q0→q2)
4. q2 now holds the original state

Here q0 is initialised to |1⟩ with an X gate, so q2 must always be measured as 1.

In [ ]:
qr = QuantumRegister(3, 'q')
cr = ClassicalRegister(3, 'c')
tele = QuantumCircuit(qr, cr)

tele.x(0)                       # state to teleport: |1>
tele.barrier()
tele.h(1); tele.cx(1, 2)        # Bell pair between q1 and q2
tele.cx(0, 1); tele.h(0)        # Alice: Bell-basis change
tele.barrier()
tele.measure([0, 1], [0, 1])    # Alice measures q0, q1
tele.barrier()
tele.cx(1, 2); tele.cz(0, 2)    # Bob's corrections
tele.measure(2, 2)              # read Bob's qubit

display(tele.draw(output='mpl'))

In [ ]:
counts = run(tele, shots=1000)
print(counts)
# Bit order is c2 c1 c0. c2 (leftmost) must be 1 for every outcome; c1 c0 are uniformly random.
assert all(k[0] == '1' for k in counts), "teleportation failed"
print("c2 = 1 in every shot -> |1> was teleported to q2")
plot_histogram(counts)

**Check with an arbitrary state** (not just |1⟩): teleport Ry(θ)|0⟩, then undo the rotation on Bob's qubit – the result must always be 0.
Here the classically-controlled corrections are done as controlled gates (deferred measurement).

In [ ]:
theta = 1.1
t2 = QuantumCircuit(3, 1)
t2.ry(theta, 0)                  # arbitrary state to teleport
t2.h(1); t2.cx(1, 2)
t2.cx(0, 1); t2.h(0)
t2.cx(1, 2); t2.cz(0, 2)         # corrections
t2.ry(-theta, 2)                 # undo preparation on Bob's side
t2.measure(2, 0)
print(run(t2, shots=2000))       # expect {'0': 2000}